In [30]:
import os


In [31]:
!pip install openai

In [32]:
import os
os.environ["OPENAI_API_KEY"] = "your_api_key_here"

In [33]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key: ··········


In [34]:
from openai import OpenAI
client = OpenAI()

In [35]:
!pip install -q yfinance pandas nest_asyncio

In [36]:
import nest_asyncio
import yfinance as yf
import pandas as pd
import numpy as np

nest_asyncio.apply()

TICKERS = ["AAPL", "AMZN", "GOOGL", "META", "NFLX", "NVDA"]

def get_yahoo_data(tickers=TICKERS, period="5y", interval="1d"):
    raw = yf.download(
        tickers=tickers,
        period=period,
        interval=interval,
        group_by="ticker",
        auto_adjust=True,
        threads=True,
        progress=False
    )

    data = {}
    for ticker in tickers:
        try:
            df = raw[ticker].copy() if len(tickers) > 1 else raw.copy()
        except:
            continue
        df = df.dropna()

        price_col = "Adj Close" if "Adj Close" in df.columns else "Close"
        if price_col not in df.columns or df.empty:
            continue

        df["Return"] = df[price_col].pct_change()
        df["Month"] = df.index.month

        monthly_seasonality = df.groupby("Month")["Return"].mean().to_dict()
        recent = df[[price_col]].tail(20).reset_index()
        recent["Date"] = recent["Date"].dt.strftime("%Y-%m-%d")

        # Volatility as std dev of returns
        volatility = df["Return"].std() * np.sqrt(252)

        # 5-year total return
        five_year_return = df[price_col].iloc[-1] / df[price_col].iloc[0] - 1

        data[ticker] = {
            "price_col": price_col,
            "recent_prices": recent.to_dict(orient="records"),
            "monthly_seasonality": monthly_seasonality,
            "latest_close": float(df[price_col].iloc[-1]),
            "one_year_return": float(df[price_col].iloc[-1] / df[price_col].iloc[-252] - 1) if len(df) >= 252 else None,
            "volatility": volatility,
            "five_year_return": five_year_return
        }
    return data

def generate_dynamic_report(data):
    report = []
    report.append("# FAANG + NVIDIA Investment Research Report")
    report.append("### Data Source: Yahoo Finance (last 5 years daily)")
    report.append("### Prepared by: Automated Analysis System\n")

    # Executive Summary
    report.append("## Executive Summary")
    avg_1y_return = np.mean([v["one_year_return"] for v in data.values() if v["one_year_return"] is not None])
    report.append(f"Over the last year, the average return across FAANG + NVIDIA was {avg_1y_return:.2%}. "
                  f"The analysis below highlights key trends, seasonality patterns, and risk considerations.\n")

    # Data Insights
    report.append("## Data Insights")
    for ticker, info in data.items():
        report.append(f"### {ticker}")
        report.append(f"- Latest closing price: ${info['latest_close']:.2f}")
        if info["one_year_return"] is not None:
            report.append(f"- Approx. 1-year return: {info['one_year_return']:.2%}")
        report.append(f"- 5-year total return: {info['five_year_return']:.2%}")
        report.append(f"- Volatility (annualized std dev): {info['volatility']:.2%}")
        report.append("- Recent 5 closing prices:")
        price_col_for_report = info.get('price_col', 'Adj Close')
        for row in info["recent_prices"][-5:]:
            report.append(f"  - {row['Date']}: ${row[price_col_for_report]:.2f}")
        report.append("")

    # Seasonality
    report.append("## Seasonality Analysis")
    for ticker, info in data.items():
        sorted_season = sorted(info["monthly_seasonality"].items(), key=lambda x: x[1], reverse=True)
        strongest_month, strongest_return = sorted_season[0]
        weakest_month, weakest_return = sorted_season[-1]
        report.append(f"### {ticker}")
        report.append(f"- Strongest month: {strongest_month} ({strongest_return:.2%})")
        report.append(f"- Weakest month: {weakest_month} ({weakest_return:.2%})\n")

    # Risk Assessment
    report.append("## Risk Assessment")
    high_vol = [ticker for ticker, v in data.items() if v["volatility"] > 0.035]
    report.append(f"- Higher-risk stocks (annualized vol > 3.5%): {', '.join(high_vol) if high_vol else 'None'}")
    report.append("- Concentration risk due to tech-heavy allocation")
    report.append("- Macro and AI regulatory risks particularly relevant to NVIDIA, Meta, and Alphabet\n")

    # Portfolio Strategy
    report.append("## Portfolio Strategy Recommendations")
    growth_stocks = [ticker for ticker, v in data.items() if v["five_year_return"] > 2]  # >200% in 5 years
    stable_stocks = [ticker for ticker in data if ticker not in growth_stocks]
    report.append(f"- Growth-focused allocation: {', '.join(growth_stocks)}")
    report.append(f"- Core/stable allocation: {', '.join(stable_stocks)}")
    report.append("- Use strongest seasonal months for tactical entries")
    report.append("- Rebalance periodically based on volatility and returns\n")

    # Long-term Thesis
    report.append("## 10-Year Investment Thesis")
    report.append(
        "Based on historical performance, FAANG + NVIDIA are likely to remain dominant in technology and AI infrastructure. "
        "A diversified approach balancing growth and stable tech stocks, guided by seasonality and risk metrics, can maximize long-term returns while mitigating downside exposure."
    )

    return "\n".join(report)

# Generate dynamic report
data = get_yahoo_data()
dynamic_report = generate_dynamic_report(data)

print(dynamic_report)

# FAANG + NVIDIA Investment Research Report
### Data Source: Yahoo Finance (last 5 years daily)
### Prepared by: Automated Analysis System

## Executive Summary
Over the last year, the average return across FAANG + NVIDIA was 27.13%. The analysis below highlights key trends, seasonality patterns, and risk considerations.

## Data Insights
### AAPL
- Latest closing price: $248.96
- Approx. 1-year return: 16.17%
- 5-year total return: 107.15%
- Volatility (annualized std dev): 27.46%
- Recent 5 closing prices:
  - 2026-03-13: $250.12
  - 2026-03-16: $252.82
  - 2026-03-17: $254.23
  - 2026-03-18: $249.94
  - 2026-03-19: $248.96

### AMZN
- Latest closing price: $208.76
- Approx. 1-year return: 6.76%
- 5-year total return: 34.21%
- Volatility (annualized std dev): 35.22%
- Recent 5 closing prices:
  - 2026-03-13: $207.67
  - 2026-03-16: $211.74
  - 2026-03-17: $215.20
  - 2026-03-18: $209.87
  - 2026-03-19: $208.76

### GOOGL
- Latest closing price: $307.13
- Approx. 1-year return: 88.05%

Pulling it all together

In [37]:
!pip install ecos

In [39]:
# -----------------------------
# 3️⃣ Enhanced Dashboard Agent
# -----------------------------
def dashboard_agent_enhanced(data, mpt_weights, decisions):
    report = []
    report.append("# Portfolio Dashboard")

    # Portfolio Allocation
    report.append("## Suggested Portfolio Allocation (MPT-based, diversified)")
    for ticker, weight in mpt_weights.items():
        report.append(f"- {ticker}: {weight:.2%}")

    # AI Decision Engine
    report.append("\n## AI Decision Engine Recommendations (with MPT context)")
    for ticker, info in decisions.items():
        mpt_weight = mpt_weights.get(ticker, 0)
        vol = data[ticker]["volatility"]
        report.append(f"### {ticker}")
        report.append(f"- Action: {info['action']}")
        report.append(f"- Confidence: {info['confidence']*100:.1f}%")
        report.append(f"- Suggested allocation (AI): {info['suggested_allocation']*100:.1f}%")
        report.append(f"- MPT allocation: {mpt_weight*100:.1f}%")
        report.append(f"- Volatility: {vol:.2%}")
        report.append(f"- Best entry month (seasonality): {info['strongest_month']}\n")

    # Risk Summary
    high_vol_threshold = 0.25  # 25% annualized
    high_vol = [t for t, v in data.items() if v["volatility"] > high_vol_threshold]
    report.append("## Risk Summary")
    if high_vol:
        report.append(f"- High volatility stocks (>25% annualized): {', '.join(high_vol)}")
    else:
        report.append("- No stocks exceed high volatility threshold.")

    # Quarterly Recommendations
    report.append("\n## Quarterly Tactical Recommendations")
    for ticker, info in decisions.items():
        report.append(
            f"- {ticker}: {info['action']} {info['suggested_allocation']*100:.1f}% allocation "
            f"(Confidence {info['confidence']*100:.1f}%, Volatility {data[ticker]['volatility']:.2%}) "
            f"if price trends align with seasonality in month {info['strongest_month']}"
        )

    return "\n".join(report)

In [40]:
import pandas as pd
import numpy as np
import cvxpy as cp

# -----------------------------
# 1️⃣ Stable & Diversified MPT Agent
# -----------------------------
def mpt_agent_diversified(data, gamma=0.5, min_weight=0.05, max_weight=0.4):
    tickers = list(data.keys())
    dfs = []

    # Prepare return series
    for ticker in tickers:
        df = pd.DataFrame(data[ticker]["recent_prices"])
        price_col = data[ticker]["price_col"]
        df["Return"] = df[price_col].pct_change()
        df = df.dropna(subset=["Return"])
        df = df[["Date", "Return"]]
        dfs.append(df.set_index("Date"))

    # Align dates
    common_index = dfs[0].index
    for df in dfs[1:]:
        common_index = common_index.intersection(df.index)

    returns = np.array([df.loc[common_index, "Return"].values for df in dfs])
    mean_returns = np.nanmean(returns, axis=1)
    cov_matrix = np.cov(returns) + np.eye(len(tickers)) * 1e-6  # regularization

    n = len(tickers)
    w = cp.Variable(n)

    # Maximize risk-adjusted return
    objective = cp.Maximize(mean_returns @ w - gamma * cp.quad_form(w, cov_matrix))
    constraints = [
        cp.sum(w) == 1,
        w >= min_weight,
        w <= max_weight
    ]

    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.SCS, verbose=False)

    weights_raw = np.maximum(w.value, 0)
    if weights_raw.sum() == 0 or weights_raw is None:
        weights_normalized = np.ones(len(weights_raw)) / len(weights_raw)
    else:
        weights_normalized = weights_raw / weights_raw.sum()

    return {tickers[i]: float(weights_normalized[i]) for i in range(n)}

# -----------------------------
# 2️⃣ Decision Engine Agent
# -----------------------------
def decision_engine_stable(data, portfolio_weights):
    decisions = {}
    total_confidence = 0
    temp_confidence = {}

    for ticker, info in data.items():
        recent_return = info["one_year_return"] or 0
        vol = info["volatility"]
        strongest_month = max(info["monthly_seasonality"], key=info["monthly_seasonality"].get)

        if recent_return > 0.1 and vol < 0.35:
            action = "Buy"
            confidence = min(0.9, recent_return * 2)
        elif recent_return < -0.05:
            action = "Sell"
            confidence = min(0.8, abs(recent_return) * 2)
        else:
            action = "Hold"
            confidence = 0.5

        temp_confidence[ticker] = confidence
        total_confidence += confidence

        decisions[ticker] = {
            "action": action,
            "confidence": confidence,
            "strongest_month": strongest_month
        }

    # Scale allocation proportional to confidence
    for ticker in decisions:
        allocation = temp_confidence[ticker] / total_confidence if total_confidence > 0 else 0
        decisions[ticker]["suggested_allocation"] = round(allocation, 2)

    return decisions

# -----------------------------
# 3️⃣ Enhanced Dashboard Agent
# -----------------------------
def dashboard_agent_enhanced(data, mpt_weights, decisions):
    report = []
    report.append("# Portfolio Dashboard")

    # Portfolio Allocation
    report.append("## Suggested Portfolio Allocation (MPT-based, diversified)")
    for ticker, weight in mpt_weights.items():
        report.append(f"- {ticker}: {weight:.2%}")

    # AI Decision Engine
    report.append("\n## AI Decision Engine Recommendations (with MPT context)")
    for ticker, info in decisions.items():
        mpt_weight = mpt_weights.get(ticker, 0)
        vol = data[ticker]["volatility"]
        report.append(f"### {ticker}")
        report.append(f"- Action: {info['action']}")
        report.append(f"- Confidence: {info['confidence']*100:.1f}%")
        report.append(f"- Suggested allocation (AI): {info['suggested_allocation']*100:.1f}%")
        report.append(f"- MPT allocation: {mpt_weight*100:.1f}%")
        report.append(f"- Volatility: {vol:.2%}")
        report.append(f"- Best entry month (seasonality): {info['strongest_month']}\n")

    # Risk Summary
    high_vol_threshold = 0.25
    high_vol = [t for t, v in data.items() if v["volatility"] > high_vol_threshold]
    report.append("## Risk Summary")
    report.append(f"- High volatility stocks (>25% annualized): {', '.join(high_vol) if high_vol else 'None'}")

    # Quarterly Recommendations
    report.append("\n## Quarterly Tactical Recommendations")
    for ticker, info in decisions.items():
        report.append(
            f"- {ticker}: {info['action']} {info['suggested_allocation']*100:.1f}% allocation "
            f"(Confidence {info['confidence']*100:.1f}%, Volatility {data[ticker]['volatility']:.2%}) "
            f"if price trends align with seasonality in month {info['strongest_month']}"
        )

    return "\n".join(report)

# -----------------------------
# 4️⃣ Run Full Pipeline
# -----------------------------
mpt_weights = mpt_agent_diversified(data)  # gamma=0.5, min/max enforced
decisions = decision_engine_stable(data, mpt_weights)
dashboard = dashboard_agent_enhanced(data, mpt_weights, decisions)

print(dashboard)

# Portfolio Dashboard
## Suggested Portfolio Allocation (MPT-based, diversified)
- AAPL: 5.00%
- AMZN: 40.00%
- GOOGL: 5.00%
- META: 5.00%
- NFLX: 40.00%
- NVDA: 5.00%

## AI Decision Engine Recommendations (with MPT context)
### AAPL
- Action: Buy
- Confidence: 32.3%
- Suggested allocation (AI): 10.0%
- MPT allocation: 5.0%
- Volatility: 27.46%
- Best entry month (seasonality): 7

### AMZN
- Action: Hold
- Confidence: 50.0%
- Suggested allocation (AI): 16.0%
- MPT allocation: 40.0%
- Volatility: 35.22%
- Best entry month (seasonality): 7

### GOOGL
- Action: Buy
- Confidence: 90.0%
- Suggested allocation (AI): 28.0%
- MPT allocation: 5.0%
- Volatility: 30.68%
- Best entry month (seasonality): 7

### META
- Action: Hold
- Confidence: 50.0%
- Suggested allocation (AI): 16.0%
- MPT allocation: 5.0%
- Volatility: 43.52%
- Best entry month (seasonality): 1

### NFLX
- Action: Hold
- Confidence: 50.0%
- Suggested allocation (AI): 16.0%
- MPT allocation: 40.0%
- Volatility: 42.95%
- Best ent

Optimize for specific days

In [41]:
import pandas as pd
import numpy as np
import cvxpy as cp

# -----------------------------
# 1️⃣ Stable & Diversified MPT Agent
# -----------------------------
def mpt_agent_diversified(data, gamma=0.5, min_weight=0.05, max_weight=0.4):
    tickers = list(data.keys())
    dfs = []

    for ticker in tickers:
        df = pd.DataFrame(data[ticker]["recent_prices"])
        price_col = data[ticker]["price_col"]
        df["Return"] = df[price_col].pct_change()
        df = df.dropna(subset=["Return"])
        df = df[["Date", "Return"]]
        dfs.append(df.set_index("Date"))

    # Align dates
    common_index = dfs[0].index
    for df in dfs[1:]:
        common_index = common_index.intersection(df.index)

    returns = np.array([df.loc[common_index, "Return"].values for df in dfs])
    mean_returns = np.nanmean(returns, axis=1)
    cov_matrix = np.cov(returns) + np.eye(len(tickers)) * 1e-6  # regularization

    n = len(tickers)
    w = cp.Variable(n)

    objective = cp.Maximize(mean_returns @ w - gamma * cp.quad_form(w, cov_matrix))
    constraints = [
        cp.sum(w) == 1,
        w >= min_weight,
        w <= max_weight
    ]

    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.SCS, verbose=False)

    weights_raw = np.maximum(w.value, 0)
    if weights_raw.sum() == 0 or weights_raw is None:
        weights_normalized = np.ones(len(weights_raw)) / len(weights_raw)
    else:
        weights_normalized = weights_raw / weights_raw.sum()

    return {tickers[i]: float(weights_normalized[i]) for i in range(n)}

# -----------------------------
# 2️⃣ Decision Engine with Buy Dates & Holding Period
# -----------------------------
def decision_engine_dates(data, portfolio_weights, hold_days=30):
    decisions = {}
    total_confidence = 0
    temp_confidence = {}

    for ticker, info in data.items():
        recent_return = info["one_year_return"] or 0
        vol = info["volatility"]

        # Best entry day = day of month with max return in historical data
        df = pd.DataFrame(info["recent_prices"])
        price_col = info["price_col"]
        df["Return"] = df[price_col].pct_change()
        df = df.dropna(subset=["Return"])
        best_idx = df["Return"].idxmax()
        best_date = df.loc[best_idx, "Date"] if len(df) > 0 else None

        if recent_return > 0.1 and vol < 0.35:
            action = "Buy"
            confidence = min(0.9, recent_return * 2)
        elif recent_return < -0.05:
            action = "Sell"
            confidence = min(0.8, abs(recent_return) * 2)
        else:
            action = "Hold"
            confidence = 0.5

        temp_confidence[ticker] = confidence
        total_confidence += confidence

        decisions[ticker] = {
            "action": action,
            "confidence": confidence,
            "best_date": best_date,
            "hold_days": hold_days
        }

    # Scale allocation proportional to confidence
    for ticker in decisions:
        allocation = temp_confidence[ticker] / total_confidence if total_confidence > 0 else 0
        decisions[ticker]["suggested_allocation"] = round(allocation, 2)

    return decisions

# -----------------------------
# 3️⃣ Dashboard with Dates & Holding Period
# -----------------------------
def dashboard_agent_dates(data, mpt_weights, decisions):
    report = []
    report.append("# Portfolio Dashboard")

    # Portfolio Allocation
    report.append("## Suggested Portfolio Allocation (MPT-based, diversified)")
    for ticker, weight in mpt_weights.items():
        report.append(f"- {ticker}: {weight:.2%}")

    # AI Decision Engine
    report.append("\n## AI Decision Engine Recommendations (with MPT context)")
    for ticker, info in decisions.items():
        mpt_weight = mpt_weights.get(ticker, 0)
        vol = data[ticker]["volatility"]
        report.append(f"### {ticker}")
        report.append(f"- Action: {info['action']}")
        report.append(f"- Confidence: {info['confidence']*100:.1f}%")
        report.append(f"- Suggested allocation (AI): {info['suggested_allocation']*100:.1f}%")
        report.append(f"- MPT allocation: {mpt_weight*100:.1f}%")
        report.append(f"- Volatility: {vol:.2%}")
        report.append(f"- Best entry date: {info['best_date']}")
        report.append(f"- Suggested holding period: {info['hold_days']} days\n")

    # Risk Summary
    high_vol_threshold = 0.25
    high_vol = [t for t, v in data.items() if v["volatility"] > high_vol_threshold]
    report.append("## Risk Summary")
    report.append(f"- High volatility stocks (>25% annualized): {', '.join(high_vol) if high_vol else 'None'}")

    # Quarterly Recommendations
    report.append("\n## Quarterly Tactical Recommendations")
    for ticker, info in decisions.items():
        report.append(
            f"- {ticker}: {info['action']} {info['suggested_allocation']*100:.1f}% allocation "
            f"(Confidence {info['confidence']*100:.1f}%, Volatility {data[ticker]['volatility']:.2%}) "
            f"buy on {info['best_date']} and hold for {info['hold_days']} days"
        )

    return "\n".join(report)

# -----------------------------
# 4️⃣ Run Full Pipeline
# -----------------------------
mpt_weights = mpt_agent_diversified(data)
decisions = decision_engine_dates(data, mpt_weights, hold_days=30)
dashboard = dashboard_agent_dates(data, mpt_weights, decisions)

print(dashboard)

# Portfolio Dashboard
## Suggested Portfolio Allocation (MPT-based, diversified)
- AAPL: 5.00%
- AMZN: 40.00%
- GOOGL: 5.00%
- META: 5.00%
- NFLX: 40.00%
- NVDA: 5.00%

## AI Decision Engine Recommendations (with MPT context)
### AAPL
- Action: Buy
- Confidence: 32.3%
- Suggested allocation (AI): 10.0%
- MPT allocation: 5.0%
- Volatility: 27.46%
- Best entry date: 2026-02-24
- Suggested holding period: 30 days

### AMZN
- Action: Hold
- Confidence: 50.0%
- Suggested allocation (AI): 16.0%
- MPT allocation: 40.0%
- Volatility: 35.22%
- Best entry date: 2026-03-04
- Suggested holding period: 30 days

### GOOGL
- Action: Buy
- Confidence: 90.0%
- Suggested allocation (AI): 28.0%
- MPT allocation: 5.0%
- Volatility: 30.68%
- Best entry date: 2026-03-09
- Suggested holding period: 30 days

### META
- Action: Hold
- Confidence: 50.0%
- Suggested allocation (AI): 16.0%
- MPT allocation: 5.0%
- Volatility: 43.52%
- Best entry date: 2026-03-16
- Suggested holding period: 30 days

### NFLX
- Act

In [ ]:
Exact days to hold optimisation

In [42]:
import pandas as pd
import numpy as np
import cvxpy as cp

# -----------------------------
# 1️⃣ Diversified MPT Agent
# -----------------------------
def mpt_agent_diversified(data, gamma=0.5, min_weight=0.05, max_weight=0.4):
    tickers = list(data.keys())
    dfs = []

    for ticker in tickers:
        df = pd.DataFrame(data[ticker]["recent_prices"])
        price_col = data[ticker]["price_col"]
        df["Return"] = df[price_col].pct_change()
        df = df.dropna(subset=["Return"])
        df = df[["Date", "Return"]]
        dfs.append(df.set_index("Date"))

    # Align dates
    common_index = dfs[0].index
    for df in dfs[1:]:
        common_index = common_index.intersection(df.index)

    returns = np.array([df.loc[common_index, "Return"].values for df in dfs])
    mean_returns = np.nanmean(returns, axis=1)
    cov_matrix = np.cov(returns) + np.eye(len(tickers)) * 1e-6  # regularization

    n = len(tickers)
    w = cp.Variable(n)
    objective = cp.Maximize(mean_returns @ w - gamma * cp.quad_form(w, cov_matrix))
    constraints = [
        cp.sum(w) == 1,
        w >= min_weight,
        w <= max_weight
    ]
    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.SCS, verbose=False)

    weights_raw = np.maximum(w.value, 0)
    weights_normalized = weights_raw / weights_raw.sum() if weights_raw.sum() > 0 else np.ones(n) / n
    return {tickers[i]: float(weights_normalized[i]) for i in range(n)}

# -----------------------------
# 2️⃣ Decision Engine with Optimal Holding Period
# -----------------------------
def decision_engine_optimal_hold(data, portfolio_weights, max_hold_days=180):
    decisions = {}
    total_confidence = 0
    temp_confidence = {}

    for ticker, info in data.items():
        recent_return = info["one_year_return"] or 0
        vol = info["volatility"]

        # Convert recent_prices to DataFrame
        df = pd.DataFrame(info["recent_prices"])
        price_col = info["price_col"]
        df["Return"] = df[price_col].pct_change()
        df = df.dropna(subset=["Return"])

        # Find best entry date
        best_idx = df["Return"].idxmax() if len(df) > 0 else None
        best_date = df.loc[best_idx, "Date"] if best_idx is not None else None

        # Simulate all possible holding periods
        optimal_hold = 1
        max_cum_return = -np.inf
        for h in range(1, min(max_hold_days, len(df)-best_idx)):
            cum_return = df[price_col].iloc[best_idx:best_idx+h].pct_change().add(1).prod() - 1
            if cum_return > max_cum_return:
                max_cum_return = cum_return
                optimal_hold = h

        # Decide action & confidence
        if recent_return > 0.1 and vol < 0.35:
            action = "Buy"
            confidence = min(0.9, recent_return * 2)
        elif recent_return < -0.05:
            action = "Sell"
            confidence = min(0.8, abs(recent_return) * 2)
        else:
            action = "Hold"
            confidence = 0.5

        temp_confidence[ticker] = confidence
        total_confidence += confidence

        decisions[ticker] = {
            "action": action,
            "confidence": confidence,
            "best_date": best_date,
            "hold_days": optimal_hold
        }

    # Scale allocation proportional to confidence
    for ticker in decisions:
        allocation = temp_confidence[ticker] / total_confidence if total_confidence > 0 else 0
        decisions[ticker]["suggested_allocation"] = round(allocation, 2)

    return decisions

# -----------------------------
# 3️⃣ Dashboard Agent
# -----------------------------
def dashboard_agent_dates(data, mpt_weights, decisions):
    report = []
    report.append("# Portfolio Dashboard")

    # Portfolio Allocation
    report.append("## Suggested Portfolio Allocation (MPT-based, diversified)")
    for ticker, weight in mpt_weights.items():
        report.append(f"- {ticker}: {weight:.2%}")

    # AI Decision Engine
    report.append("\n## AI Decision Engine Recommendations (with MPT context)")
    for ticker, info in decisions.items():
        mpt_weight = mpt_weights.get(ticker, 0)
        vol = data[ticker]["volatility"]
        report.append(f"### {ticker}")
        report.append(f"- Action: {info['action']}")
        report.append(f"- Confidence: {info['confidence']*100:.1f}%")
        report.append(f"- Suggested allocation (AI): {info['suggested_allocation']*100:.1f}%")
        report.append(f"- MPT allocation: {mpt_weight*100:.1f}%")
        report.append(f"- Volatility: {vol:.2%}")
        report.append(f"- Best entry date: {info['best_date']}")
        report.append(f"- Optimal holding period: {info['hold_days']} days\n")

    # Risk Summary
    high_vol_threshold = 0.25
    high_vol = [t for t, v in data.items() if v["volatility"] > high_vol_threshold]
    report.append("## Risk Summary")
    report.append(f"- High volatility stocks (>25% annualized): {', '.join(high_vol) if high_vol else 'None'}")

    # Quarterly Recommendations
    report.append("\n## Quarterly Tactical Recommendations")
    for ticker, info in decisions.items():
        report.append(
            f"- {ticker}: {info['action']} {info['suggested_allocation']*100:.1f}% allocation "
            f"(Confidence {info['confidence']*100:.1f}%, Volatility {data[ticker]['volatility']:.2%}) "
            f"buy on {info['best_date']} and hold for {info['hold_days']} days"
        )

    return "\n".join(report)

# -----------------------------
# 4️⃣ Run Full Pipeline
# -----------------------------
mpt_weights = mpt_agent_diversified(data)
decisions = decision_engine_optimal_hold(data, mpt_weights, max_hold_days=180)
dashboard = dashboard_agent_dates(data, mpt_weights, decisions)

print(dashboard)

# Portfolio Dashboard
## Suggested Portfolio Allocation (MPT-based, diversified)
- AAPL: 5.00%
- AMZN: 40.00%
- GOOGL: 5.00%
- META: 5.00%
- NFLX: 40.00%
- NVDA: 5.00%

## AI Decision Engine Recommendations (with MPT context)
### AAPL
- Action: Buy
- Confidence: 32.3%
- Suggested allocation (AI): 10.0%
- MPT allocation: 5.0%
- Volatility: 27.46%
- Best entry date: 2026-02-24
- Optimal holding period: 1 days

### AMZN
- Action: Hold
- Confidence: 50.0%
- Suggested allocation (AI): 16.0%
- MPT allocation: 40.0%
- Volatility: 35.22%
- Best entry date: 2026-03-04
- Optimal holding period: 1 days

### GOOGL
- Action: Buy
- Confidence: 90.0%
- Suggested allocation (AI): 28.0%
- MPT allocation: 5.0%
- Volatility: 30.68%
- Best entry date: 2026-03-09
- Optimal holding period: 6 days

### META
- Action: Hold
- Confidence: 50.0%
- Suggested allocation (AI): 16.0%
- MPT allocation: 5.0%
- Volatility: 43.52%
- Best entry date: 2026-03-16
- Optimal holding period: 1 days

### NFLX
- Action: Hold
- 

Suggestions

In [43]:
# Install if needed:
# !pip install plotly pandas

import pandas as pd
import plotly.express as px

# -----------------------------
# 1️⃣ Prepare Data
# -----------------------------
report_data = [
    {"Ticker":"AAPL", "Action":"Buy", "Confidence":32.6, "AI Allocation":10.0, "MPT Allocation":5.0,
     "Volatility":27.48, "Best Date":"2026-02-24", "Hold Days":1},
    {"Ticker":"AMZN", "Action":"Hold", "Confidence":50.0, "AI Allocation":16.0, "MPT Allocation":40.0,
     "Volatility":35.21, "Best Date":"2026-03-04", "Hold Days":1},
    {"Ticker":"GOOGL", "Action":"Buy", "Confidence":90.0, "AI Allocation":28.0, "MPT Allocation":5.0,
     "Volatility":30.67, "Best Date":"2026-03-09", "Hold Days":6},
    {"Ticker":"META", "Action":"Hold", "Confidence":50.0, "AI Allocation":16.0, "MPT Allocation":5.0,
     "Volatility":43.51, "Best Date":"2026-03-16", "Hold Days":1},
    {"Ticker":"NFLX", "Action":"Hold", "Confidence":50.0, "AI Allocation":16.0, "MPT Allocation":40.0,
     "Volatility":42.95, "Best Date":"2026-02-27", "Hold Days":4},
    {"Ticker":"NVDA", "Action":"Hold", "Confidence":50.0, "AI Allocation":16.0, "MPT Allocation":5.0,
     "Volatility":51.66, "Best Date":"2026-03-02", "Hold Days":7},
]

df = pd.DataFrame(report_data)
df["Best Date"] = pd.to_datetime(df["Best Date"])
df["End Date"] = df["Best Date"] + pd.to_timedelta(df["Hold Days"], unit="d")

# -----------------------------
# 2️⃣ Portfolio Allocation Comparison
# -----------------------------
fig_alloc = px.bar(df, x="Ticker", y=["AI Allocation","MPT Allocation"],
                   barmode="group",
                   color_discrete_sequence=px.colors.qualitative.Bold,
                   title="Portfolio Allocation: AI vs MPT")
fig_alloc.update_layout(yaxis_title="Allocation (%)")
fig_alloc.show()

# -----------------------------
# 3️⃣ Confidence vs Volatility
# -----------------------------
fig_scatter = px.scatter(df, x="Volatility", y="Confidence", color="Action",
                         size="AI Allocation", text="Ticker",
                         hover_data=["Best Date","Hold Days"],
                         color_discrete_sequence=px.colors.qualitative.Vivid,
                         title="Confidence vs Volatility")
fig_scatter.update_layout(xaxis_title="Volatility (%)", yaxis_title="Confidence (%)")
fig_scatter.show()

# -----------------------------
# 4️⃣ Best Entry Date & Holding Period Timeline
# -----------------------------
fig_timeline = px.timeline(df, x_start="Best Date", x_end="End Date", y="Ticker",
                           color="Action", text="Hold Days",
                           color_discrete_sequence=px.colors.qualitative.Pastel,
                           title="Best Entry Dates & Optimal Holding Periods")
fig_timeline.update_yaxes(autorange="reversed")  # Tickers top-down
fig_timeline.update_layout(xaxis_title="Date", yaxis_title="Ticker", showlegend=True)
fig_timeline.show()

# -----------------------------
# ✅ Save HTML Output (optional)
# -----------------------------
fig_alloc.write_html("portfolio_allocation.html")
fig_scatter.write_html("confidence_vs_volatility.html")
fig_timeline.write_html("holding_period_timeline.html")

# **Monthly Seasonality Heatmap** **(Further Visualizations)**

In [44]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# ── Build seasonality matrix ──────────────────────────────────────────────────
month_names = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]

tickers = list(data.keys())
matrix = []

for ticker in tickers:
    monthly = data[ticker]["monthly_seasonality"]
    row = [monthly.get(m, 0) * 100 for m in range(1, 13)]  # convert to %
    matrix.append(row)

z_matrix = np.array(matrix)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig_heatmap = go.Figure(data=go.Heatmap(
    z=z_matrix,
    x=month_names,
    y=tickers,
    colorscale="RdYlGn",          # red = negative, green = positive
    zmid=0,                        # center color scale at 0
    text=np.round(z_matrix, 2),
    texttemplate="%{text}%",
    textfont={"size": 11},
    colorbar=dict(title="Avg Monthly Return (%)")
))

fig_heatmap.update_layout(
    title="📅 Monthly Seasonality Heatmap — Avg Return by Month (%)",
    xaxis_title="Month",
    yaxis_title="Ticker",
    height=400,
    font=dict(size=12)
)

fig_heatmap.show()

# **Radar/Spider Chart (Multi-Metric Comparison) Visualization**

In [45]:
import plotly.graph_objects as go

# ── Normalize metrics to 0–100 scale for radar ───────────────────────────────
tickers = list(data.keys())

one_yr_returns  = [max(data[t]["one_year_return"] or 0, 0) * 100 for t in tickers]
five_yr_returns = [max(data[t]["five_year_return"], 0) * 100         for t in tickers]
volatilities    = [data[t]["volatility"] * 100                       for t in tickers]
confidences     = [decisions[t]["confidence"] * 100                  for t in tickers]

def normalize(values):
    mn, mx = min(values), max(values)
    if mx == mn:
        return [50] * len(values)
    return [round((v - mn) / (mx - mn) * 100, 1) for v in values]

categories = ["1-Year Return", "5-Year Return",
              "Low Volatility", "AI Confidence", "1-Year Return"]  # repeat first to close shape

colors = ["#636EFA","#EF553B","#00CC96","#AB63FA","#FFA15A","#19D3F3"]

fig_radar = go.Figure()

norm_1yr  = normalize(one_yr_returns)
norm_5yr  = normalize(five_yr_returns)
norm_vol  = normalize([-v for v in volatilities])   # invert: lower vol = better
norm_conf = normalize(confidences)

for i, ticker in enumerate(tickers):
    values = [
        norm_1yr[i],
        norm_5yr[i],
        norm_vol[i],
        norm_conf[i],
        norm_1yr[i]   # close the polygon
    ]
    fig_radar.add_trace(go.Scatterpolar(
        r=values,
        theta=categories,
        fill="toself",
        name=ticker,
        line=dict(color=colors[i], width=2),
        opacity=0.65
    ))

fig_radar.update_layout(
    title="🕸️ Multi-Metric Radar Chart — Stock Comparison (Normalized 0–100)",
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 100])
    ),
    showlegend=True,
    height=550,
    font=dict(size=12)
)

fig_radar.show()

# **Return vs Risk Bubble Chart with MPT Weights**

In [46]:
import plotly.express as px
import pandas as pd

# ── Assemble data ─────────────────────────────────────────────────────────────
rows = []
for ticker in data:
    rows.append({
        "Ticker":         ticker,
        "1Y Return (%)":  round((data[ticker]["one_year_return"] or 0) * 100, 2),
        "Volatility (%)": round(data[ticker]["volatility"] * 100, 2),
        "5Y Return (%)":  round(data[ticker]["five_year_return"] * 100, 2),
        "MPT Weight (%)": round(mpt_weights[ticker] * 100, 1),
        "Action":         decisions[ticker]["action"],
        "Confidence (%)": round(decisions[ticker]["confidence"] * 100, 1)
    })

df_bubble = pd.DataFrame(rows)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig_bubble = px.scatter(
    df_bubble,
    x="Volatility (%)",
    y="1Y Return (%)",
    size="MPT Weight (%)",
    color="Action",
    text="Ticker",
    hover_data=["5Y Return (%)","Confidence (%)","MPT Weight (%)"],
    color_discrete_map={"Buy": "#00CC96", "Hold": "#FFA15A", "Sell": "#EF553B"},
    size_max=60,
    title="💹 Risk-Return Bubble Chart — Bubble Size = MPT Portfolio Weight"
)

# Add quadrant reference lines
fig_bubble.add_hline(y=0,  line_dash="dash", line_color="gray", opacity=0.5)
fig_bubble.add_vline(x=df_bubble["Volatility (%)"].mean(),
                     line_dash="dash", line_color="gray", opacity=0.5,
                     annotation_text="Avg Volatility",
                     annotation_position="top right")

fig_bubble.update_traces(textposition="top center", textfont_size=12)
fig_bubble.update_layout(
    xaxis_title="Annualized Volatility (%)",
    yaxis_title="1-Year Return (%)",
    height=520,
    font=dict(size=12),
    legend_title="AI Decision"
)

fig_bubble.show()

# **Rolling 90-Day Volatility Over Time (All Tickers)**

In [47]:
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# ── Fetch raw price data ──────────────────────────────────────────────────────
TICKERS = ["AAPL", "AMZN", "GOOGL", "META", "NFLX", "NVDA"]

raw = yf.download(
    tickers=TICKERS,
    period="5y",
    interval="1d",
    group_by="ticker",
    auto_adjust=True,
    threads=True,
    progress=False
)

colors = {
    "AAPL":  "#636EFA",
    "AMZN":  "#EF553B",
    "GOOGL": "#00CC96",
    "META":  "#AB63FA",
    "NFLX":  "#FFA15A",
    "NVDA":  "#19D3F3"
}

# ── Build figure ──────────────────────────────────────────────────────────────
fig_rolling_vol = go.Figure()

for ticker in TICKERS:
    try:
        df = raw[ticker].copy()
    except KeyError:
        continue

    price_col = "Close"
    df = df[[price_col]].dropna()
    df["DailyReturn"] = df[price_col].pct_change()

    # 90-day rolling annualized volatility
    df["RollingVol"] = df["DailyReturn"].rolling(window=90).std() * np.sqrt(252) * 100

    fig_rolling_vol.add_trace(go.Scatter(
        x=df.index,
        y=df["RollingVol"],
        name=ticker,
        mode="lines",
        line=dict(color=colors[ticker], width=1.8),
        hovertemplate=f"<b>{ticker}</b><br>Date: %{{x|%b %d, %Y}}<br>Rolling Vol: %{{y:.1f}}%<extra></extra>"
    ))

# ── Add a risk threshold reference line ──────────────────────────────────────
fig_rolling_vol.add_hline(
    y=35,
    line_dash="dot",
    line_color="red",
    opacity=0.6,
    annotation_text="High Risk Threshold (35%)",
    annotation_position="bottom right",
    annotation_font_color="red"
)

fig_rolling_vol.update_layout(
    title="📈 Rolling 90-Day Annualized Volatility — 5-Year Window",
    xaxis_title="Date",
    yaxis_title="Annualized Volatility (%)",
    hovermode="x unified",
    height=520,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    font=dict(size=12),
    xaxis=dict(
        rangeselector=dict(
            buttons=[
                dict(count=6,  label="6M", step="month", stepmode="backward"),
                dict(count=1,  label="1Y", step="year",  stepmode="backward"),
                dict(count=3,  label="3Y", step="year",  stepmode="backward"),
                dict(step="all", label="5Y")
            ]
        ),
        rangeslider=dict(visible=True),
        type="date"
    )
)

fig_rolling_vol.show()

# **Monthly Seasonality Box Plot (Return Distribution by Month)**

In [48]:
import yfinance as yf
import pandas as pd
import plotly.express as px

# ── Fetch & reshape data ──────────────────────────────────────────────────────
TICKERS = ["AAPL", "AMZN", "GOOGL", "META", "NFLX", "NVDA"]

raw = yf.download(
    tickers=TICKERS,
    period="5y",
    interval="1d",
    group_by="ticker",
    auto_adjust=True,
    threads=True,
    progress=False
)

month_map = {
    1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr", 5:"May",  6:"Jun",
    7:"Jul", 8:"Aug", 9:"Sep", 10:"Oct", 11:"Nov", 12:"Dec"
}

all_rows = []

for ticker in TICKERS:
    try:
        df = raw[ticker].copy()
    except KeyError:
        continue

    price_col = "Close"
    df = df[[price_col]].dropna()
    df["MonthlyReturn"] = df[price_col].resample("ME").last().pct_change() * 100

    monthly_df = df["MonthlyReturn"].dropna().reset_index()
    monthly_df.columns = ["Date", "MonthlyReturn"]
    monthly_df["Month"]      = monthly_df["Date"].dt.month
    monthly_df["MonthLabel"] = monthly_df["Month"].map(month_map)
    monthly_df["Ticker"]     = ticker

    all_rows.append(monthly_df)

df_all = pd.concat(all_rows, ignore_index=True)

# Preserve calendar month ordering
month_order = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]
df_all["MonthLabel"] = pd.Categorical(
    df_all["MonthLabel"], categories=month_order, ordered=True
)
df_all = df_all.sort_values("MonthLabel")

# ── Dropdown: one box plot per ticker ─────────────────────────────────────────
colors_seq = ["#636EFA","#EF553B","#00CC96","#AB63FA","#FFA15A","#19D3F3"]

fig_season = px.box(
    df_all,
    x="MonthLabel",
    y="MonthlyReturn",
    color="Ticker",
    facet_col="Ticker",
    facet_col_wrap=3,
    color_discrete_sequence=colors_seq,
    points="all",          # show individual data points as dots
    hover_data=["Date"],
    title="📅 Monthly Return Distribution by Ticker — 5 Years of Seasonality"
)

# ── Zero return reference line on each facet ──────────────────────────────────
fig_season.add_hline(
    y=0,
    line_dash="dash",
    line_color="gray",
    opacity=0.5
)

fig_season.update_layout(
    yaxis_title="Monthly Return (%)",
    height=720,
    showlegend=False,
    font=dict(size=11)
)

fig_season.update_xaxes(tickangle=45, title_text="")
fig_season.update_yaxes(title_text="Monthly Return (%)")

fig_season.show()

# **Correlation Network Graph (Stock Interdependency Map)**

In [49]:
import yfinance as yf
import pandas as pd
import numpy as np
import networkx as nx
import plotly.graph_objects as go

# ── Fetch & compute correlation matrix ───────────────────────────────────────
TICKERS = ["AAPL", "AMZN", "GOOGL", "META", "NFLX", "NVDA"]

raw = yf.download(TICKERS, period="5y", interval="1d",
                  group_by="ticker", auto_adjust=True,
                  threads=True, progress=False)

returns = pd.DataFrame()
for ticker in TICKERS:
    try:
        df = raw[ticker]["Close"].pct_change().dropna()
        returns[ticker] = df
    except:
        continue

corr_matrix = returns.corr()

# ── Build network graph ───────────────────────────────────────────────────────
G = nx.Graph()
G.add_nodes_from(TICKERS)

CORR_THRESHOLD = 0.5   # only draw edges above this correlation

for i in range(len(TICKERS)):
    for j in range(i + 1, len(TICKERS)):
        t1, t2 = TICKERS[i], TICKERS[j]
        corr_val = corr_matrix.loc[t1, t2]
        if corr_val >= CORR_THRESHOLD:
            G.add_edge(t1, t2, weight=round(corr_val, 3))

# ── Spring layout for node positions ─────────────────────────────────────────
pos = nx.spring_layout(G, seed=42, k=2.5)

# ── Build edge traces ─────────────────────────────────────────────────────────
edge_traces = []
for u, v, d in G.edges(data=True):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    weight  = d["weight"]
    width   = (weight - CORR_THRESHOLD) / (1 - CORR_THRESHOLD) * 8 + 1  # scale width

    edge_traces.append(go.Scatter(
        x=[x0, x1, None],
        y=[y0, y1, None],
        mode="lines",
        line=dict(width=width, color=f"rgba(99,110,250,{weight:.2f})"),
        hoverinfo="text",
        text=f"{u} ↔ {v}: ρ = {weight}",
        showlegend=False
    ))

# ── Build node traces ─────────────────────────────────────────────────────────
node_x, node_y, node_text, node_hover = [], [], [], []
node_sizes  = []
node_colors = []

color_map = {"AAPL":"#636EFA","AMZN":"#EF553B","GOOGL":"#00CC96",
             "META":"#AB63FA","NFLX":"#FFA15A","NVDA":"#19D3F3"}

for ticker in G.nodes():
    x, y = pos[ticker]
    degree = G.degree(ticker)
    node_x.append(x)
    node_y.append(y)
    node_text.append(ticker)
    node_sizes.append(25 + degree * 12)   # bigger = more connections
    node_colors.append(color_map[ticker])
    node_hover.append(
        f"<b>{ticker}</b><br>Connections: {degree}<br>"
        + "<br>".join([
            f"ρ({ticker},{nbr}) = {G[ticker][nbr]['weight']}"
            for nbr in G.neighbors(ticker)
        ])
    )

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode="markers+text",
    text=node_text,
    textposition="top center",
    hovertext=node_hover,
    hoverinfo="text",
    marker=dict(
        size=node_sizes,
        color=node_colors,
        line=dict(width=2, color="white")
    )
)

# ── Assemble figure ───────────────────────────────────────────────────────────
fig_network = go.Figure(data=edge_traces + [node_trace])

fig_network.update_layout(
    title="🕸️ Stock Correlation Network — Edge Width & Opacity = Correlation Strength",
    showlegend=False,
    hovermode="closest",
    height=580,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    font=dict(size=12),
    annotations=[dict(
        text="Threshold: ρ ≥ 0.50 | Node size = number of strong correlations",
        showarrow=False, xref="paper", yref="paper",
        x=0.5, y=-0.05, font=dict(size=10, color="gray")
    )]
)

fig_network.show()

# **STL Decomposition: Trend + Seasonality + Residual**

In [50]:
!pip install -q statsmodels

import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import STL

# ── Fetch data ────────────────────────────────────────────────────────────────
TICKERS = ["AAPL", "AMZN", "GOOGL", "META", "NFLX", "NVDA"]

raw = yf.download(TICKERS, period="5y", interval="1d",
                  group_by="ticker", auto_adjust=True,
                  threads=True, progress=False)

colors = {"AAPL":"#636EFA","AMZN":"#EF553B","GOOGL":"#00CC96",
          "META":"#AB63FA","NFLX":"#FFA15A","NVDA":"#19D3F3"}

# ── Create 6x3 subplot grid (ticker × component) ──────────────────────────────
fig_stl = make_subplots(
    rows=6, cols=3,
    subplot_titles=[
        f"{t} — {comp}"
        for t in TICKERS
        for comp in ["Trend", "Seasonality", "Residual"]
    ],
    vertical_spacing=0.04,
    horizontal_spacing=0.06
)

for row_idx, ticker in enumerate(TICKERS, start=1):
    try:
        df = raw[ticker]["Close"].dropna()
    except:
        continue

    # Resample to weekly to smooth & speed up STL
    df_weekly = df.resample("W").last().dropna()

    # STL decomposition — period=52 for annual seasonality in weekly data
    stl = STL(df_weekly, period=52, robust=True)
    result = stl.fit()

    dates     = df_weekly.index
    trend     = result.trend
    seasonal  = result.seasonal
    residual  = result.resid
    color     = colors[ticker]

    # ── Trend ────────────────────────────────────────────────────────────────
    fig_stl.add_trace(go.Scatter(
        x=dates, y=trend,
        mode="lines", name=f"{ticker} Trend",
        line=dict(color=color, width=1.8),
        showlegend=(row_idx == 1),
        hovertemplate="%{x|%b %Y}: $%{y:.2f}<extra></extra>"
    ), row=row_idx, col=1)

    # ── Seasonality ──────────────────────────────────────────────────────────
    fig_stl.add_trace(go.Scatter(
        x=dates, y=seasonal,
        mode="lines", name=f"{ticker} Season",
        line=dict(color=color, width=1.2, dash="dot"),
        showlegend=False,
        hovertemplate="%{x|%b %Y}: %{y:.3f}<extra></extra>"
    ), row=row_idx, col=2)

    # ── Residual — bar chart for noise ────────────────────────────────────────
    fig_stl.add_trace(go.Bar(
        x=dates, y=residual,
        name=f"{ticker} Residual",
        marker_color=np.where(
            np.array(residual) >= 0, color, "#FF6B6B"
        ),
        showlegend=False,
        hovertemplate="%{x|%b %Y}: %{y:.3f}<extra></extra>"
    ), row=row_idx, col=3)

fig_stl.update_layout(
    title="🔬 STL Decomposition — Trend / Seasonality / Residual per Stock (Weekly, 5Y)",
    height=1800,
    font=dict(size=10),
    hovermode="x unified"
)

fig_stl.show()

# **Underwater (Drawdown) Chart with Recovery Bands**

In [51]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Fetch data ────────────────────────────────────────────────────────────────
TICKERS = ["AAPL", "AMZN", "GOOGL", "META", "NFLX", "NVDA"]

raw = yf.download(TICKERS, period="5y", interval="1d",
                  group_by="ticker", auto_adjust=True,
                  threads=True, progress=False)

colors = {"AAPL":"#636EFA","AMZN":"#EF553B","GOOGL":"#00CC96",
          "META":"#AB63FA","NFLX":"#FFA15A","NVDA":"#19D3F3"}

# ── Build 6-row subplot figure ─────────────────────────────────────────────────
fig_dd = make_subplots(
    rows=6, cols=1,
    shared_xaxes=True,
    subplot_titles=[f"{t} — Drawdown from Peak" for t in TICKERS],
    vertical_spacing=0.04
)

for row_idx, ticker in enumerate(TICKERS, start=1):
    try:
        df = raw[ticker]["Close"].dropna()
    except:
        continue

    # Rolling peak & drawdown
    rolling_max = df.cummax()
    drawdown    = (df - rolling_max) / rolling_max * 100  # in %

    # Find worst drawdown point
    min_dd_idx = drawdown.idxmin()
    min_dd_val = drawdown.min()

    color = colors[ticker]

    # ── Filled drawdown area ──────────────────────────────────────────────────
    fig_dd.add_trace(go.Scatter(
        x=df.index,
        y=drawdown.values,
        mode="lines",
        fill="tozeroy",
        fillcolor=f"rgba({int(color[1:3],16)},{int(color[3:5],16)},{int(color[5:7],16)},0.18)",
        line=dict(color=color, width=1.5),
        name=ticker,
        showlegend=True,
        hovertemplate=f"<b>{ticker}</b><br>%{{x|%b %d, %Y}}<br>Drawdown: %{{y:.1f}}%<extra></extra>"
    ), row=row_idx, col=1)

    # ── Worst drawdown annotation ─────────────────────────────────────────────
    fig_dd.add_annotation(
        x=min_dd_idx,
        y=min_dd_val,
        text=f"Max DD: {min_dd_val:.1f}%",
        showarrow=True,
        arrowhead=2,
        arrowcolor=color,
        font=dict(size=9, color=color),
        bgcolor="white",
        bordercolor=color,
        row=row_idx, col=1
    )

    # ── -20% "Bear Market" reference line ─────────────────────────────────────
    fig_dd.add_hline(
        y=-20,
        line_dash="dot",
        line_color="red",
        opacity=0.4,
        row=row_idx, col=1
    )

    # ── Shade recovery periods (drawdown < -20%) ──────────────────────────────
    in_bear = drawdown < -20
    start   = None
    for date, flag in in_bear.items():
        if flag and start is None:
            start = date
        elif not flag and start is not None:
            fig_dd.add_vrect(
                x0=start, x1=date,
                fillcolor="rgba(255,0,0,0.07)",
                layer="below", line_width=0,
                row=row_idx, col=1
            )
            start = None
    if start is not None:   # still in drawdown at end of window
        fig_dd.add_vrect(
            x0=start, x1=df.index[-1],
            fillcolor="rgba(255,0,0,0.07)",
            layer="below", line_width=0,
            row=row_idx, col=1
        )

fig_dd.update_layout(
    title="📉 Underwater Drawdown Chart — Drawdown from Rolling All-Time High (5Y)",
    height=1400,
    hovermode="x unified",
    font=dict(size=11),
    legend=dict(orientation="h", yanchor="bottom", y=1.01,
                xanchor="right", x=1)
)

fig_dd.update_yaxes(ticksuffix="%")

fig_dd.show()